# Task 3 — Real-World Mini Project: GitHub Profile Analyzer

**Topic:** Combining everything — `requests` + real public API + error handling + a small CLI-style tool

**Kya seekhenge (what we'll learn):**
- Ek REAL production API (GitHub) ke saath kaam karna, mock API nahi
- Rate limiting aur `403` errors handle karna
- Multiple endpoints combine kar ke ek "report" banana
- Data ko process karna (sort, filter, aggregate)
- Result ko local JSON file mein save karna
- Ek chhota interactive tool banana jo GitHub project ke liye perfect hai

**API used:** `https://api.github.com` — public, koi authentication zaroori nahi (lekin bina auth ke
sirf 60 requests/hour milti hain — isliye is notebook mein hum requests ginti se use karenge).

> Ye notebook Task 1 aur Task 2 se thoda aage hai — is mein hum sab kuch mila kar
> ek chhota lekin "real" tool banayenge jo GitHub pe publish karne ke qabil ho.


In [ ]:
import requests
import json
from datetime import datetime

GITHUB_API = "https://api.github.com"

# GitHub thodi si extra info deta hai agar hum User-Agent header bhejein — good practice hai
HEADERS = {"User-Agent": "requests-practice-notebook"}

print("Ready!")


## 1. Ek User Ka Profile Fetch Karna

GitHub API se kisi bhi public user ka data mil sakta hai — koi login/token zaroori nahi.


In [ ]:
username = "torvalds"   # Linux ke banane wale — GitHub pe available hain

response = requests.get(f"{GITHUB_API}/users/{username}", headers=HEADERS)
print("Status code:", response.status_code)

user_data = response.json()
print("Poora data ek dictionary hai, kuch important keys:")
print(list(user_data.keys())[:10])   # sirf pehli 10 keys dikha rahe hain


## 2. Sirf Zaroori Info Nikal Kar Print Karna

In [ ]:
def print_profile(user_data):
    """Dictionary se sirf important fields nikal kar sundar tareeqe se print karta hai."""
    print(f"Name         : {user_data.get('name')}")
    print(f"Username     : {user_data.get('login')}")
    print(f"Bio          : {user_data.get('bio')}")
    print(f"Followers    : {user_data.get('followers')}")
    print(f"Following    : {user_data.get('following')}")
    print(f"Public Repos : {user_data.get('public_repos')}")
    print(f"Profile URL  : {user_data.get('html_url')}")

print_profile(user_data)


## 3. Rate Limits Check Karna

GitHub har response ke headers mein batata hai ke aapke paas kitni requests baaki hain.
Ye production APIs ke saath kaam karte waqt bohat zaroori habit hai.


In [ ]:
remaining = response.headers.get("X-RateLimit-Remaining")
limit = response.headers.get("X-RateLimit-Limit")
reset_time = response.headers.get("X-RateLimit-Reset")

print(f"Baaki requests: {remaining} / {limit}")

if reset_time:
    reset_dt = datetime.fromtimestamp(int(reset_time))
    print(f"Limit is waqt reset hogi: {reset_dt}")


## 4. User Ke Repos Fetch Karna (Query Parameters Ke Saath)

GitHub API `sort` aur `per_page` jese query parameters accept karti hai.


In [ ]:
params = {
    "sort": "updated",   # sabse recent update wale repos pehle
    "per_page": 10        # sirf 10 repos mangwao
}

response = requests.get(f"{GITHUB_API}/users/{username}/repos", params=params, headers=HEADERS)
repos = response.json()

# NOTE: agar rate limit khatam ho jaye ya koi aur error aaye, GitHub ek LIST ki jagah
# ek dictionary bhejta hai (jese {"message": "API rate limit exceeded", ...}).
# Isliye hamesha type check karna acha practice hai.
if isinstance(repos, list):
    print(f"Total {len(repos)} repos mile.\n")
    for repo in repos[:5]:
        print(f"- {repo['name']}  (⭐ {repo['stargazers_count']})")
else:
    print("Repos nahi mil sake — shayad rate limit lag gayi hai. Server ka jawab:")
    print(repos.get("message", repos))


## 5. Data Process Karna — Stars Ke Hisaab Se Sort Karna

Ab jo data mila hai usko Python mein process karte hain — ye asli "engineering" wala hissa hai,
sirf API call karna kaafi nahi hota.


In [ ]:
# stargazers_count ke hisaab se sort kar rahe hain, sabse zyada stars wala pehle
# (agar upar wale cell mein rate limit lagi thi to repos khali list []  bana lete hain,
#  taake ye cell crash na ho)
if not isinstance(repos, list):
    repos = []

top_repos = sorted(repos, key=lambda r: r["stargazers_count"], reverse=True)

print("Top 5 repos (by stars):\n")
for repo in top_repos[:5]:
    print(f"{repo['name']:<25} ⭐ {repo['stargazers_count']:<6} 🍴 {repo['forks_count']}")

if not top_repos:
    print("(Koi repo data nahi mila is waqt — upar wala cell dobara chalao ya thodi der baad try karo.)")


## 6. Error Handling — User Na Mile To Kya Karein

Agar username galat ho to GitHub `404` deta hai. Rate limit khatam ho jaye to `403`.
Dono cases handle karna zaroori hai.


In [ ]:
def get_github_user(username):
    """
    Safe way se GitHub user data fetch karta hai.
    Success par dictionary return karta hai, warna None.
    """
    response = requests.get(f"{GITHUB_API}/users/{username}", headers=HEADERS)

    if response.status_code == 200:
        return response.json()
    elif response.status_code == 404:
        print(f"User '{username}' nahi mila (404).")
        return None
    elif response.status_code == 403:
        print("Rate limit khatam ho gayi hai (403) — thodi der baad try karo.")
        return None
    else:
        print(f"Kuch aur error aaya: {response.status_code}")
        return None


# Test: ek valid aur ek invalid username
valid = get_github_user("torvalds")
invalid = get_github_user("this-user-should-not-exist-12345")

print()
print("Valid result mila:", valid is not None)
print("Invalid result mila:", invalid is not None)


## 7. Result Ko JSON File Mein Save Karna

Real projects mein aksar data ko save karte hain taake baar baar API call na karni pade.


In [ ]:
report = {
    "username": username,
    "fetched_at": str(datetime.now()),
    "profile": {
        "name": user_data.get("name"),
        "followers": user_data.get("followers"),
        "public_repos": user_data.get("public_repos"),
    },
    "top_repos": [
        {"name": r["name"], "stars": r["stargazers_count"]}
        for r in top_repos[:5]
    ]
}

with open("github_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Report save ho gayi: github_report.json")
print(json.dumps(report, indent=2))


---
## Exercises — Ab Aapki Bari

Har `# TODO` ki jagah apna code likho.


### Exercise 1 — Apna GitHub Profile Fetch Karo
`get_github_user()` function use kar ke apna khud ka GitHub username fetch karo aur `print_profile()` se print karo.

In [ ]:
# TODO: apna GitHub username daal kar get_github_user() call karo
# TODO: agar data mile to print_profile() se print karo



### Exercise 2 — Language Breakdown
User ke repos mein se har repo ki `language` field nikalo aur count karo ke kis language mein kitne repos hain (dictionary use karo: `{'Python': 3, 'JavaScript': 2, ...}`).

In [ ]:
# TODO: language_count dictionary banao {}
# TODO: repos list mein loop chalao, har repo ki 'language' field check karo
# TODO: agar language None na ho to count badhao
# TODO: final dictionary print karo



### Exercise 3 — Total Stars Calculate Karo
Saare repos ke `stargazers_count` ko add kar ke total stars nikalo (ek line mein `sum()` se ho sakta hai).

In [ ]:
# TODO: total stars calculate karo (sum + list comprehension use kar sakte ho)
# TODO: print karo "Total stars across all repos: X"



### Exercise 4 — Mini Project: Repo Search Tool (Thoda Mushkil)

GitHub ka Search API bhi hai: `https://api.github.com/search/repositories?q=<query>&sort=stars`

Ek function likho `search_top_repos(topic, count=5)` jo:
1. Given `topic` (jese `"machine-learning"`) ke liye repos search kare
2. Stars ke hisaab se sort kiye hue top `count` repos return kare (list of dicts: name, stars, url)
3. Agar koi error aaye (403 rate limit waghera) to empty list return kare aur error print kare

Phir isko `topic="fastapi"` ke saath test karo aur result print karo.

**Hint:** Search API response mein repos `response.json()["items"]` ke andar hote hain, seedha list nahi.


In [ ]:
# TODO: search_top_repos function define karo
def search_top_repos(topic, count=5):
    pass  # apna code yahan likho


# TODO: "fastapi" topic ke saath test karo aur result print karo


---
## Wrap-up + Publishing Checklist

Is notebook mein humne seekha:
- Real production API (GitHub) ke saath kaam karna
- Rate limits check karna aur respect karna
- Multiple endpoints se data combine kar ke ek report banana
- Data process karna (sort, count, sum)
- Result ko file mein save karna
- Proper error handling (404, 403, aur generic errors)

### GitHub Pe Publish Karne Se Pehle:
1. Apna naam aur project ka short description README.md mein likho
2. `.gitignore` mein `github_report.json` add kar sakte ho agar wo generated file rakhni nahi
3. Har exercise solve karne ke baad notebook ko "Run All" kar ke check karo koi error to nahi
4. Repo ka naam kuch clear rakho, jese: `github-profile-analyzer` ya `requests-library-practice`
